# Tier 2 — the Unknown branch: where is the record too weak to trust?

The third Tier-2 branch off the Tier-1 coarse allocator ([`06_analysis.ipynb`](06_analysis.ipynb)).
Tier 1 predicts an Unknown share — the burned area with **no attributed cause** — as a class in its
own right (18.5% of all acres). This notebook treats that Unknown mass not as a fire forecast but as
a **data-quality signal**: for a region-season, how much of the burned area is un-attributable, and
is that missingness *stable* enough to tell a planner in advance **where their own cause inputs are
unreliable**. The deliverable is therefore an **operational recommendation** — where to invest in
cause reporting — not a prediction of fire.

**Two questions:**
1. **Is data-quality predictable?** Can next season's `missing_acre_frac` be anticipated from a
   region-season's own history (so the planner is warned before the season, not after)?
2. **Where does missingness concentrate — and does it track Natural burn?** This was worked through
   in **W3** already ([`03_missingness.ipynb`](03_missingness.ipynb) and the W3 status report): three
   confound-controlled probes found **no evidence** the Missing bucket selectively hides Natural
   fires, and the residual "West" concern was scoped down to a *precision* caveat (recent western
   cells the forecast leans on), not a Natural-vs-human bias. This branch **confirms that W3 result
   at a new grain** — the cross-sectional level of missing share across ecoregions — and uses it to
   settle whether the resolved **Human 22.7% is a floor**.

**Method.** Same grain, boundary rule, and forward-chaining split as the rest of the pipeline.
Cells run top-to-bottom, left **unexecuted** for manual run.

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from config import ProjectConfig
from panel import RegionSeasonPanel

# Project-wide constants (paths, the boundary rule, the forward-chaining split) come
# from one place so this branch cannot drift from the others. See src/config.py.
cfg = ProjectConfig()

# RegionSeasonPanel loads the analysis grain, applies the partial-winter boundary rule,
# and derives each branch's target. `attribution_quality()` gives one row per
# region-season with the missing-cause fraction plus the coarse Human/Natural acres, so
# data quality can be related to the Natural share. The per-cell missing_* columns are
# DEDUPLICATED, not summed -- they repeat across the cell's 12 cause rows, and summing
# would inflate the Unknown mass 12x.
panel = RegionSeasonPanel.load(cfg)
cell = panel.attribution_quality()

print(f"{len(cell):,} region-season cells")
print(f"missing_acre_frac: mean {cell.missing_acre_frac.mean():.3f}, "
      f"median {cell.missing_acre_frac.median():.3f}")

10,135 region-season cells
missing_acre_frac: mean 0.238, median 0.118


## Q1 — Is data-quality predictable in advance?

If a region-season's missing-cause fraction is stable year to year, the planner can be warned
*before* the season that their attribution will be weak there. Same forward-chaining machinery as
the other branches: predict `missing_acre_frac` from the trailing mean of the cell's own prior
same-season occurrences (k=7), scored on the held-out tail (≥ 2010) with MAE, against a global-mean
baseline. `missing_acre_frac` is already a bounded [0,1] fraction, so MAE in raw fraction points is
the natural metric (no log transform).

In [2]:
TEST_START = cfg.test_start     # forward-chaining split, shared across all branches
K = cfg.shares_k                # trailing window locked by the Tier-1 shares sweep

# The forward-chaining rule now lives in src/trailing.py rather than being re-typed
# per notebook. TrailingMean does shift(1) then a k-window mean within
# (region, season), and asserts the frame is sorted first -- on an unsorted frame the
# raw idiom silently attaches one region's history to another's rows.
from trailing import GlobalPrior, TrailingMean

pred_trail = TrailingMean(K).predict(cell, "missing_acre_frac")["missing_acre_frac"].to_numpy()

actual = cell["missing_acre_frac"].to_numpy()
w = cell["total_ac"].to_numpy()
in_test = (cell["season_year"] >= TEST_START).to_numpy()
train = (cell["season_year"] < TEST_START).to_numpy()

def mae(pred):
    err = np.abs(pred - actual)
    m = in_test & ~np.isnan(err)
    return {"n_cells": int(m.sum()),
            "MAE_unwtd": float(err[m].mean()),
            "MAE_acre_wtd": float(np.average(err[m], weights=w[m]))}

# Reference: one constant for every cell, fit on training years only (no region info).
pred_global = (GlobalPrior(weighted=False)
               .fit(cell, "missing_acre_frac", train_mask=train)
               .predict(cell)["missing_acre_frac"].to_numpy())

res_q1 = pd.DataFrame([mae(pred_global), mae(pred_trail)],
                      index=["global mean", f"persistence (k={K})"])
print(f"missing_acre_frac -- held-out tail season_year >= {TEST_START}\n")
print(res_q1.round(4).to_string())
print("\nlower MAE = better. Persistence beating the global mean = data-quality is region-season")
print("specific and forecastable, so the 'weak-attribution' warning can be issued pre-season.")

missing_acre_frac -- held-out tail season_year >= 2010

                   n_cells  MAE_unwtd  MAE_acre_wtd
global mean           3954     0.2230        0.2398
persistence (k=7)     3949     0.2012        0.1665

lower MAE = better. Persistence beating the global mean = data-quality is region-season
specific and forecastable, so the 'weak-attribution' warning can be issued pre-season.


## Q2 — Does missingness track the high-Natural West? (confirming the W3 correction at a new grain)

**W3 already answered the bias question** ([`03_missingness.ipynb`](03_missingness.ipynb)): three
confound-controlled probes — compositional stability detrended, within region×season, and cause-mix
by size — found **no evidence** the Missing bucket selectively hides Natural fires (Natural r ≈ −0.02
detrended, ≈ +0.02 within-cell). The size-selectivity it *does* show lands on the small, human,
Debris-heavy end, not on Natural. The only survivor was a narrow **precision** caveat: high-missing
cells are disproportionately recent western region-seasons the forecast leans on.

Here we look from a different angle — not rate-*change* or within-cell detrending, but the plain
cross-sectional **level**: across ecoregions, does a higher Natural share go with a higher missing
share? If the "high-Natural West absorbs Natural into Missing" picture were right, we'd see a
**positive** correlation. A **negative** one is consistent with W3 and pins missingness to
human-dominated regions instead.

In [3]:
# Aggregate to the ecoregion (the design claim is about regions, not individual cells): total
# Natural / Human / missing acres over the full record, then shares.
reg = cell.groupby("region").agg(nat=("Natural", "sum"), hum=("Human", "sum"),
                                 miss=("missing_acres", "sum")).reset_index()
reg["tot"] = reg["nat"] + reg["hum"] + reg["miss"]
reg = reg[reg["tot"] > 0].copy()
reg["nat_share"] = reg["nat"] / reg["tot"]
reg["miss_share"] = reg["miss"] / reg["tot"]

pear = reg["nat_share"].corr(reg["miss_share"])
spear = reg["nat_share"].corr(reg["miss_share"], method="spearman")
print(f"across {len(reg)} ecoregions -- corr(Natural share, missing share):")
print(f"   Pearson  {pear:+.3f}")
print(f"   Spearman {spear:+.3f}\n")

print("Highest-missing ecoregions (where attribution is weakest):")
print(reg.nlargest(8, "miss_share")[["region", "miss_share", "nat_share"]].round(2).to_string(index=False))
print("\nLowest-missing ecoregions:")
print(reg.nsmallest(8, "miss_share")[["region", "miss_share", "nat_share"]].round(2).to_string(index=False))

across 105 ecoregions -- corr(Natural share, missing share):
   Pearson  -0.636
   Spearman -0.558

Highest-missing ecoregions (where attribution is weakest):
                  region  miss_share  nat_share
    Central Great Plains        0.69       0.05
   Southern Texas Plains        0.64       0.01
 Southwestern Tablelands        0.63       0.20
             Flint Hills        0.59       0.02
         Edwards Plateau        0.58       0.19
             Coast Range        0.53       0.27
             High Plains        0.53       0.20
Eastern Corn Belt Plains        0.52       0.01

Lowest-missing ecoregions:
                  region  miss_share  nat_share
        Aleutian Islands         0.0       0.00
    Arctic Coastal Plain         0.0       0.99
       Ogilvie Mountains         0.0       1.00
      Wrangell Mountains         0.0       1.00
        Arctic Foothills         0.0       1.00
             Yukon Flats         0.0       1.00
            Brooks Range         0.0       0.

## The picture holds — missingness sits in human regions, not the high-Natural West

**Confirm on run, but the dry-run is unambiguous:** at the ecoregion level the correlation between
Natural share and missing share is **strongly negative** (Pearson ≈ −0.64). Missingness does **not**
concentrate in the high-Natural West — it concentrates in **low-Natural, human-dominated** regions:
the highest-missing ecoregions are the Central Great Plains, Southern Texas Plains, Flint Hills and
Edwards Plateau (Natural share ≈ 1–20%), while the near-zero-missing regions are the high-Natural
Alaskan/Arctic ecoregions (Natural share ≈ 100%).

**This confirms the W3 conclusion at a new grain — it does not overturn a live assumption.** By W3
the absorption hypothesis had already been retired: three probes in `03_missingness.ipynb` found no
evidence the bucket hides Natural, and the West concern was carried forward only as a *precision*
weight, not a bias claim. The cross-sectional −0.64 here is a fresh, independent line of evidence
pointing the same way: whatever drives non-attribution, it is an agency/practice axis in
human-dominated country (consistent with the mixed high-missing states — NY, CO, KS, AZ, CA — and the
BLM / ST&L reporting-stream rise `03` characterized), not Natural fire being relabeled in the West.

**The Human-floor conclusion holds — and this sharpens its reasoning.** Because unattributed acres
sit disproportionately in human regions, redistributing any of the Unknown mass onto resolved causes
would add *more* to Human than to Natural. So the resolved **Human 22.7% is a floor**, and if
anything the true Human share is **higher**. Tier 1 keeps the floor visible by predicting the Unknown
share as its own class rather than distributing it.

> **Flagged for follow-up (not changed here):** `design_refinement.md`, the W4 status report, and the
> project `CLAUDE.md` still narrate a "high-Natural West" *mechanism* as if it were live. It was
> already scoped down in W3; those docs should be reconciled to say so. This notebook reports the
> confirming measurement; the doc edits are a separate, deliberate step.

## The operational deliverable: where to invest in cause reporting

The branch's product is not a fire forecast but a **targeting list**: the region-seasons where
attribution is both **weak** (high predicted `missing_acre_frac`) and **material** (meaningful
burned area), so that improving cause reporting there would most improve the planner's own inputs.
Ranked by predicted missing acres on the held-out tail.

In [4]:
# Predicted missing acres = predicted missing fraction x total acres, on scored held-out cells.
scored = in_test & ~np.isnan(pred_trail)
out = cell[scored].copy()
out["pred_missing_frac"] = pred_trail[scored]
out["pred_missing_ac"] = out["pred_missing_frac"] * out["total_ac"]

# Aggregate over the tail to a region-season targeting list (mean predicted weakness + acres at stake).
target = (out.groupby(["region", "season"])
          .agg(mean_pred_missing_frac=("pred_missing_frac", "mean"),
               total_acres_at_stake=("total_ac", "sum"),
               pred_missing_acres=("pred_missing_ac", "sum"))
          .reset_index())

print("Where to invest in cause reporting -- top region-seasons by predicted unattributed acres")
print("(weak attribution AND material burn), held-out tail:\n")
print(target.nlargest(12, "pred_missing_acres")
      .assign(mean_pred_missing_frac=lambda d: (d.mean_pred_missing_frac * 100).round(0),
              total_acres_at_stake=lambda d: (d.total_acres_at_stake / 1e3).round(0),
              pred_missing_acres=lambda d: (d.pred_missing_acres / 1e3).round(0))
      .rename(columns={"mean_pred_missing_frac": "miss%", "total_acres_at_stake": "acres_k",
                       "pred_missing_acres": "pred_miss_k"})
      .to_string(index=False))

Where to invest in cause reporting -- top region-seasons by predicted unattributed acres
(weak attribution AND material burn), held-out tail:

                                             region season  miss%  acres_k  pred_miss_k
                            Southwestern Tablelands    MAM   46.0   2495.0       1165.0
 Central California Foothills and Coastal Mountains    JJA   36.0   2463.0        851.0
                                   Columbia Plateau    JJA   38.0   2107.0        831.0
                               Central Great Plains    MAM   57.0   1151.0        703.0
                                   Columbia Plateau    SON   59.0    796.0        538.0
                                        High Plains    MAM   43.0   1028.0        441.0
Klamath Mountains/California High North Coast Range    JJA   17.0   3006.0        415.0
                            Central Basin and Range    JJA   11.0   3297.0        395.0
                                     North Cascades    JJA   31.0

## Where this branch goes next

- **Reconcile the design docs** (flagged above): `design_refinement.md`, the W4 status report, and
  `CLAUDE.md` still narrate the "high-Natural West" mechanism as live, though W3 already scoped it to
  a precision caveat — a cross-notebook consistency edit, checking `03_missingness.ipynb`'s original
  grain first.
- **Sharpen the recommendation** into a cost-style ranking (acres-at-stake × weakness) once a
  planner cost model is available.
- **Optional learned rung** on the fingerprint features, if data-quality proves worth predicting
  more precisely than persistence — lower priority, since this branch's value is the targeting list
  and confirming the missingness picture, not forecast accuracy.